In [1]:
# install requirements
!pip install torch transformers


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

C:\Users\broba\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 429.44it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
text = '''The future of AI is'''

In [4]:
input_ids = tokenizer(text, return_tensors='pt', return_attention_mask=False)['input_ids']
output = model(input_ids=input_ids)

# Question 1
What does `output.logits` represent?


Answer : output.logits contains the raw prediction scores produced by GPT-2 for every token in the vocabulary at every position of the input sequence. These scores are not probabilities yet and are usually converted to probabilities using the softmax function.
The tensor has shape (batch_size, sequence_length, vocab_size). For each token position, GPT-2 assigns a score to every token in its vocabulary, indicating how likely that token is to appear next.

In [5]:
output.logits

tensor([[[ -36.2874,  -35.0114,  -38.0794,  ...,  -40.5164,  -41.3760,
           -34.9194],
         [ -62.3992,  -62.8435,  -69.7349,  ...,  -71.8215,  -69.3740,
           -63.9975],
         [ -78.3217,  -77.0288,  -80.2247,  ...,  -79.5169,  -82.1571,
           -77.2190],
         [ -82.0191,  -80.5210,  -86.3373,  ...,  -89.5435,  -89.3032,
           -81.6365],
         [-119.8218, -119.7811, -126.3039,  ..., -128.0584, -124.4376,
          -123.0562]]], grad_fn=<UnsafeViewBackward0>)

# Question 2
And what does `output.logits[0,-1,:]` represent?

output.logits[0,-1,:] represents the raw prediction scores for all tokens in the vocabulary for the next token after the current input sequence. The first index selects the first sequence in the batch, -1 selects the last token position, and : selects all vocabulary entries.

In [6]:
output.logits[0,-1,:]

tensor([-119.8218, -119.7811, -126.3039,  ..., -128.0584, -124.4376,
        -123.0562], grad_fn=<SelectBackward0>)

# Partial Implementation
**Suggestion**: first, use the "playground" below to correctly generate the next token. Only later add your code inside the `generate` function.

In [10]:
EOS_TOKENS = [
    198, # newline
    50256 # tokenizer.eos_token_id = '<|endoftext|>'
]

def generate(model, tokenizer, text, max_new_tokens=100, do_sample=False, top_k=None, verbose=True):
    input_ids = tokenizer(text, return_tensors='pt', return_attention_mask=False)['input_ids']
    if verbose:
        print(tokenizer.decode(input_ids[0]))
    for i in range(max_new_tokens):
        # run model
        output = model(input_ids = input_ids)

        next_token_logits = output.logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).item()

        # add next_token to the input
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]])], dim=1)
        if verbose:
            print(tokenizer.decode([next_token]))
        # stop if generated newline or eos_token (end-of-sequence)
        if next_token in EOS_TOKENS:
            break
    print(tokenizer.decode(input_ids[0]))

## Playground
Focus on predicting the next token.

In [11]:
EOS_TOKENS = [
    198, # newline
    50256 # tokenizer.eos_token_id = '<|endoftext|>'
]

input_ids = tokenizer(text, return_tensors='pt', return_attention_mask=False)['input_ids']
output = model(input_ids = input_ids)

next_token_logits = output.logits[:, -1, :]
next_token = torch.argmax(next_token_logits, dim=-1).item()

# Exercise 1 - Greedy Generation
Write the greedy generation function: always selecting the highest-scored token.
```
generate(model, tokenizer, text, max_new_tokens=20)
```
**Expected output**: The name of the president of USA is "President Obama."

In [12]:
generate(model, tokenizer, text, max_new_tokens=20)

The future of AI is
 uncertain
.
 The
 future
 of
 AI
 is
 uncertain
.


The future of AI is uncertain. The future of AI is uncertain.



# Exercise 2
Add sampling with `do_sample` parameter.
```
generate(model, tokenizer, text, max_new_tokens=20, do_sample=True)
```

## Hints
- Use [F.softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html) to obtain probabilities from logits.
- Use [torch.multinomial](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) to sample from the probabilities.

## Answer of question 2: 

In [14]:
def generate(model, tokenizer, text, max_new_tokens=100, do_sample=False, top_k=None, verbose=True):
    input_ids = tokenizer(text, return_tensors='pt', return_attention_mask=False)['input_ids']

    if verbose:
        print(tokenizer.decode(input_ids[0]))

    for i in range(max_new_tokens):

        # run model
        output = model(input_ids=input_ids)

        next_token_logits = output.logits[:, -1, :]

        if do_sample:
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
        else:
            next_token = torch.argmax(next_token_logits, dim=-1).item()

        # add next_token to the input
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]])], dim=1)

        if verbose:
            print(tokenizer.decode([next_token]))

        # stop if generated newline or eos_token
        if next_token in EOS_TOKENS:
            break

    print(tokenizer.decode(input_ids[0]))

In [15]:
# torch.manual_seed(0) # you can use this to fix the random seed
generate(model, tokenizer, text, max_new_tokens=20, do_sample=True)

The future of AI is
 being
 closely
 watched
 by
 so
 many
 malicious
 activities
,
 Rasm
us
 said
,
 one
 of
 the
 likes
 of
 which
 was
The future of AI is being closely watched by so many malicious activities, Rasmus said, one of the likes of which was


# Exercise 3
Add topk sampling with `top_k` parameter.
```
generate(model, tokenizer, text, max_new_tokens=20, top_k=10, do_sample=True)
```

## Hints
- Use [torch.topk](https://docs.pytorch.org/docs/stable/generated/torch.topk.html) to get the topk scores `.values` and topk indices `.indices`.

## Answer of exercise 3 : 

In [16]:
def generate(model, tokenizer, text, max_new_tokens=100, do_sample=False, top_k=None, verbose=True):

    input_ids = tokenizer(
        text,
        return_tensors='pt',
        return_attention_mask=False
    )['input_ids']

    if verbose:
        print(tokenizer.decode(input_ids[0]))

    for i in range(max_new_tokens):

        # run model
        output = model(input_ids=input_ids)

        next_token_logits = output.logits[:, -1, :]

        if do_sample:

            if top_k is not None:

                topk = torch.topk(
                    next_token_logits,
                    k=top_k,
                    dim=-1
                )

                probs = F.softmax(
                    topk.values,
                    dim=-1
                )

                sampled_idx = torch.multinomial(
                    probs,
                    num_samples=1
                )

                next_token = topk.indices.gather(
                    -1,
                    sampled_idx
                ).item()

            else:

                probs = F.softmax(
                    next_token_logits,
                    dim=-1
                )

                next_token = torch.multinomial(
                    probs,
                    num_samples=1
                ).item()

        else:

            next_token = torch.argmax(
                next_token_logits,
                dim=-1
            ).item()

        # add next token
        input_ids = torch.cat(
            [input_ids, torch.tensor([[next_token]])],
            dim=1
        )

        if verbose:
            print(tokenizer.decode([next_token]))

        # stop if generated newline or eos token
        if next_token in EOS_TOKENS:
            break

    print(tokenizer.decode(input_ids[0]))

In [17]:
# torch.manual_seed(0) # you can use this to fix the random seed
generate(model, tokenizer, text, max_new_tokens=20, top_k=10, do_sample=True)

The future of AI is
 in
 question
.
 We
 are
 going
 to
 have
 to
 learn
 from
 it
.
 But
 it
 may
 not
 be
 a
 good
The future of AI is in question. We are going to have to learn from it. But it may not be a good


# Exercise 4
Add temperature-based sampling with a `temperature` parameter to your generation function.

Modify the logits before applying the softmax by dividing them by the temperature value:

    p_i = softmax(z_i / T)

where $T > 0$ is the temperature.

Implement this inside the `generate` function and test it with sampling enabled mode and with different values of temperature.


In [18]:
def generate_temperature(
    model,
    tokenizer,
    text,
    max_new_tokens=20,
    do_sample=True,
    top_k=10,
    temperature=1.0,
    verbose=True
):

    input_ids = tokenizer(
        text,
        return_tensors="pt",
        return_attention_mask=False
    )["input_ids"]

    if verbose:
        print(tokenizer.decode(input_ids[0]))

    for i in range(max_new_tokens):

        output = model(input_ids=input_ids)

        next_token_logits = output.logits[:, -1, :]

        if top_k is not None:

            topk = torch.topk(
                next_token_logits,
                top_k
            )

            topk_logits = topk.values
            topk_indices = topk.indices

            probs = F.softmax(
                topk_logits / temperature,
                dim=-1
            )

            sampled_idx = torch.multinomial(
                probs,
                num_samples=1
            )

            next_token = topk_indices[
                0,
                sampled_idx.item()
            ].item()

        else:

            probs = F.softmax(
                next_token_logits / temperature,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            ).item()

        input_ids = torch.cat(
            [
                input_ids,
                torch.tensor([[next_token]])
            ],
            dim=1
        )

        if verbose:
            print(tokenizer.decode([next_token]))

        if next_token in EOS_TOKENS:
            break

    print(tokenizer.decode(input_ids[0]))

In [19]:
generate_temperature(
    model,
    tokenizer,
    text,
    max_new_tokens=20,
    top_k=10,
    temperature=0.7
)

The future of AI is
 in
 a
 new
 era
 of
 artificial
 intelligence
,
 and
 it
's
 a
 good
 one
.


The future of AI is in a new era of artificial intelligence, and it's a good one.



In [20]:
generate_temperature(
    model,
    tokenizer,
    text,
    max_new_tokens=20,
    top_k=10,
    temperature=1.0
)

The future of AI is
 a
 big
 deal
 because
 it
 could
 make
 it
 easier
 to
 create
 and
 test
 new
 technologies
 and
 to
 use
 them
 for
The future of AI is a big deal because it could make it easier to create and test new technologies and to use them for


In [21]:
generate_temperature(
    model,
    tokenizer,
    text,
    max_new_tokens=20,
    top_k=10,
    temperature=1.5
)

The future of AI is
 very
,
 very
 different
.


The future of AI is very, very different.




### Question 3
Does it make sense to apply temperature scaling in greedy decoding?

No, temperature scaling is generally not useful in greedy decoding. Greedy decoding always selects the token with the highest logit using argmax. Temperature changes the probability distribution but usually does not change the ranking of logits. Therefore, temperature mainly affects random sampling methods and has little or no effect on greedy decoding.